# Workshop 3 · 06 — Operationalisierung in Fabric

AI-Anreicherung und Data Science entfalten ihren Wert erst, wenn sie **in native Fabric-Features** münden — andernfalls bleibt es bei reinen Notebook-Analysen. Wir verbinden die Ergebnis-Tabellen mit:

1. **Gold-Tabelle** (kuratiert) als Single Source.
2. **Direct-Lake-Semantikmodell + Power BI** — Self-Service-BI ohne Datenexport.
3. **Activator** — automatische Reaktion auf kritische Befunde.
4. **SemPy** — Data-Quality auf den **AI-Ausgaben**.
5. **Data Agent** — natürlichsprachige Fragen über das angereicherte Lakehouse.

In [ ]:
from pyspark.sql.functions import col

# Kuratierte Gold-Tabelle: Stammdaten + AI-Felder + ML-Ergebnisse je Meldung
clean = spark.table("asset_clean")
ml = spark.table("meldung_ml").select("meldung_id", "cluster", "thema", "anomalie")

gold = (clean.join(ml, "meldung_id", "left")
        .select("meldung_id", "wagon_id", "baureihe_norm", "depot_norm",
                "komponente", "ort", "schweregrad", "sicherheitsrelevant",
                "prioritaet", "thema", "anomalie"))

gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("asset_gold")
display(gold)

## Data-Quality auf den AI-Ausgaben mit SemPy

Bevor AI-Ergebnisse weiterverwendet werden, prüft **SemPy** funktionale Abhängigkeiten. Beispiel — `schweregrad` sollte `prioritaet` bestimmen. Verletzungen weisen auf inkonsistente AI-Ausgaben hin.

> SemPy ist im Fabric-Runtime (Spark 3.4+) ohne Installation verfügbar.

In [ ]:
from sempy.fabric import FabricDataFrame

fdf = FabricDataFrame(spark.table("asset_gold").toPandas())

# Erwartete Abhaengigkeit: schweregrad -> prioritaet. Verletzungen = inkonsistente AI-Ausgaben.
violations = fdf.list_dependency_violations(determinant_col="schweregrad", dependent_col="prioritaet")
display(violations)

## Direct-Lake-Semantikmodell + Power BI (geführt)

Die Tabellen liegen in OneLake — Power BI liest sie per **Direct Lake** ohne Import/Refresh:

1. Lakehouse öffnen → oben rechts **New semantic model**.
2. Tabellen `asset_gold`, `kommentar_insights`, `schaden_work_orders` wählen → **Confirm**.
3. Im Modell **Maße** anlegen, z. B.
   - `Offene Auftraege = COUNTROWS(FILTER(schaden_work_orders, schaden_work_orders[dispatch] = TRUE()))`
   - `Anomalien = CALCULATE(COUNTROWS(asset_gold), asset_gold[anomalie] = TRUE())`
4. **+ New report** → Visuals: Aufträge je Werkstatt, Stimmung je Depot, Schweregrad-Verteilung, Anomalie-Liste.

**Vorteil:** Eine Kopie in OneLake = Quelle für Notebook, DS, BI und Agent — kein Sync.

## Activator — automatisch reagieren (geführt)

`schaden_work_orders` enthält `dispatch = true` für kritische Befunde. Daraus eine Aktion:

1. Aus der Tabelle `schaden_work_orders` ein **Activator**-Objekt anlegen (Real-Time Hub → Activator, oder direkt aus dem Semantikmodell).
2. Regel: **WHEN** `dispatch = true` **THEN** Teams-Nachricht an die Werkstatt-Disposition (oder E-Mail / Power Automate / Fabric-Item starten).
3. Analog: **WHEN** `anomalie = true` in `asset_gold` → Hinweis an die Qualitätssicherung.

## Data Agent — natürliche Sprache über die Daten (geführt)

1. Workspace → **New → Data agent**.
2. Datenquelle = das **Lakehouse** (Tabellen `asset_gold`, `kommentar_insights`, `schaden_work_orders`).
3. Instruktionen geben (Domäne: Wagen/Werkstatt/Depots) und Beispiel-Fragen testen:
   - „Welche Komponente verursacht die meisten sicherheitsrelevanten Meldungen?"
   - „Zeig offene Aufträge der Werkstatt Strukturprüfung."
   - „In welchem Depot ist die Stimmung am schlechtesten?"

> Tiefer Agenten-/RAG-Aufbau ist ein eigener Slot (*Fabric Agents + AI Foundry*) — hier nur die Anbindung als Operationalisierung.


### Gesamtbild

```
Rohdaten ──AI Functions──▶ saubere/strukturierte Tabellen ──Embeddings+ML──▶ meldung_ml
        │                                                                          │
        └──────────────────────────▶ asset_gold ◀──────────────────────────────────┘
                                          │
        ┌─────────────────┬───────────────┼────────────────┬─────────────────┐
   Power BI           Activator        SemPy            Data Agent        Pipelines
  (Direct Lake)     (Teams-Alert)   (Data-Quality)   (NL-Fragen)       (Orchestrierung)
```

Das ist der Unterschied zu reinen Notebook-Analysen: Jede Stufe mündet in ein natives Fabric-Feature — eine Kopie der Daten in OneLake, end-to-end.